In [1]:
import pandas as pd
## Importing Movie Lens 2M datasets

In [2]:
ratings = pd.read_csv('E:\\University Stuff\\Third year\\movie_recommender_research\\datasets\\movie_lens_2m\\ratings.csv')
movie = pd.read_csv('E:\\University Stuff\\Third year\\movie_recommender_research\\datasets\\movie_lens_2m\\movies.csv')

In [3]:
movie['year'] = movie.title.str.extract('(\\d\d\d\d\))',
expand=False)
#Removing the parentheses
movie['year'] = movie.year.str.extract('(\d\d\d\d)',expand=False)
#Removing the years from the 'title' column
movie['title'] = movie.title.str.replace('(\(\d\d\d\d\))', '')
#Applying the strip function to get rid of any ending whitespace characters that may have appeared
movie['title'] = movie['title'].apply(lambda x: x.strip())

C:\Users\Vlad\AppData\Local\Temp\ipykernel_5768\2164731462.py:6: FutureWarning: The default value of regex will change from True to False in a future version.
  movie['title'] = movie.title.str.replace('(\(\d\d\d\d\))', '')


In [4]:
movie.drop(columns=['genres'], inplace=True)

In [5]:
ratings.drop(columns=['timestamp'], inplace=True)
ratings

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5
...,...,...,...
20000258,138493,68954,4.5
20000259,138493,69526,4.5
20000260,138493,69644,3.0
20000261,138493,70286,5.0


In [6]:
movie

,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995
...,...,...,...
27273,131254,Kein Bund für's Leben,2007
27274,131256,"Feuer, Eis & Dosenbier",2002
27275,131258,The Pirates,2014
27276,131260,Rentun Ruusu,2001


## Create User input

In [7]:
user = [
            {'title':'Toy Story, The', 'rating':5},
            # {'title':'Toy Story', 'rating':2.5},
            # {'title':'Jumanji', 'rating':3},
            # {'title':"Pulp Fiction", 'rating':4.5},
            # {'title':'Akira', 'rating':5}
         ]
input_movie = pd.DataFrame(user)
input_movie
#Filtering out the movies by title
Id = movie[movie['title'].isin(input_movie['title'].tolist())]
#Then merging it so we can get the movieId. It's implicitly merging it by title.
input_movie = pd.merge(Id, input_movie)
input_movie.head()

,movieId,title,year,rating
0,1,Toy Story,1995,2.5
1,2,Jumanji,1995,3.0
2,296,Pulp Fiction,1994,4.5
3,1274,Akira,1988,5.0
4,1968,"Breakfast Club, The",1985,4.0


## Filtering users that have rated one of the given movies

In [8]:
def filter_users(given_ratings):
    #Filtering out users that have watched movies that the input has watched and storing it
    users = given_ratings[ratings['movieId'].isin(input_movie['movieId'].tolist())]
    user_subset_group = users.groupby(['userId'])
    #Sorting it so that users with movie ratings most in common with the input will have priority
    user_subset_group = sorted(user_subset_group,  key=lambda x: len(x[1]), reverse=True)
    return user_subset_group

user_subset_group = filter_users(ratings)

In [9]:
type(user_subset_group[0][1])
print(user_subset_group[0])

(91,       userId  movieId  rating
9621      91        1     4.0
9622      91        2     3.5
9669      91      296     3.5
9826      91     1274     2.5
9903      91     1968     4.0)


In [10]:
import math
#Store the Pearson Correlation in a dictionary, where the key is the user Id and the value is the coefficient
pearsonCorDict = {}
input_movie = input_movie.sort_values(by='movieId')
#For every user group in our subset
for name, group in user_subset_group:
    #Let's start by sorting the input and current user group so the values aren't mixed up later on
    group = group.sort_values(by='movieId')
    #Get the N for the formula
    n = len(group)
    #Get the review scores for the movies that they both have in common
    temp = input_movie[input_movie['movieId'].isin(group['movieId'].tolist())]
    #And then store them in a temporary buffer variable in a list format to facilitate future calculations
    tempRatingList = temp['rating'].tolist()
    #put the current user group reviews in a list format
    tempGroupList = group['rating'].tolist()
    #Now let's calculate the pearson correlation between two users, so called, x and y
    Sxx = sum([i**2 for i in tempRatingList]) - pow(sum(tempRatingList),2)/float(n)
    Syy = sum([i**2 for i in tempGroupList]) - pow(sum(tempGroupList),2)/float(n)
    Sxy = sum( i*j for i, j in zip(tempRatingList, tempGroupList)) - sum(tempRatingList)*sum(tempGroupList)/float(n)

    #If the denominator is different than zero, then divide, else, 0 correlation.
    if Sxx != 0 and Syy != 0:
        pearsonCorDict[name] = Sxy/math.sqrt(Sxx*Syy)
    else:
        pearsonCorDict[name] = 0

In [11]:
pearsonCorDict.items()

dict_items([(91, -0.6890618270883883), (294, 0.10783277320343156), (586, 0.7836445860269199), (648, 0.444102681159703), (775, 0.46266531814837414), (812, -0.12945217625467967), (869, 0.07624928516630236), (903, 0.0660338179744212), (1200, 0.2494610901255917), (1244, 0.29654012630945475), (1715, 0.6309898162000303), (1748, 0.5114957546028552), (1763, 0.1760901812651271), (1810, 0.6990252954195334), (1813, 0.36589645615870564), (1849, 0.06603381797442423), (1864, 0.5114957546028552), (1942, 0.23262521394079627), (1984, -0.7994259492812168), (2047, 0.5477103564747346), (2099, -0.10783277320343156), (2367, -0.10783277320343994), (2397, 0), (2515, 0.9244734516419062), (2661, 0.835703992326648), (2757, 0.8439249387982215), (2959, 0.23055616708169688), (2988, 0.29809064964264287), (3179, 0.0), (3218, 0.26413527189768793), (3268, 0.7781270639007126), (3269, 0.3606167767094639), (3318, 0.3026049692947228), (3397, 0.46107317554294897), (3487, -0.3774147062120338), (3576, 0.39620290784652895), (3

In [12]:
pearsonDF = pd.DataFrame.from_dict(pearsonCorDict, orient='index')
pearsonDF.columns = ['similarityIndex']
pearsonDF['userId'] = pearsonDF.index
pearsonDF.index = range(len(pearsonDF))
pearsonDF.head()


,similarityIndex,userId
0,-0.689062,91
1,0.107833,294
2,0.783645,586
3,0.444103,648
4,0.462665,775


In [13]:
topUsers=pearsonDF.sort_values(by='similarityIndex', ascending=False)[0:50]
topUsers.head()

,similarityIndex,userId
17821,1.0,114862
16902,1.0,105389
8106,1.0,11718
14910,1.0,83705
16850,1.0,104656


In [14]:
topUsersRating=topUsers.merge(ratings, left_on='userId', right_on='userId', how='inner')
topUsersRating.head()

,similarityIndex,userId,movieId,rating
0,1.0,114862,2,2.0
1,1.0,114862,9,1.0
2,1.0,114862,10,4.0
3,1.0,114862,18,4.0
4,1.0,114862,24,4.0


In [15]:
#Multiplies the similarity by the user's ratings
topUsersRating['weightedRating'] = topUsersRating['similarityIndex']*topUsersRating['rating']
topUsersRating.head()

,similarityIndex,userId,movieId,rating,weightedRating
0,1.0,114862,2,2.0,2.0
1,1.0,114862,9,1.0,1.0
2,1.0,114862,10,4.0,4.0
3,1.0,114862,18,4.0,4.0
4,1.0,114862,24,4.0,4.0


In [16]:
#Applies a sum to the topUsers after grouping it up by userId
tempTopUsersRating = topUsersRating.groupby('movieId').sum()[['similarityIndex','weightedRating']]
tempTopUsersRating.columns = ['sum_similarityIndex','sum_weightedRating']
tempTopUsersRating.head()

,sum_similarityIndex,sum_weightedRating
movieId,,
1,24.0,49.0
2,32.0,63.0
3,7.0,18.0
4,4.0,7.5
5,7.0,20.0


In [17]:
#Creates an empty dataframe
recommendation_df = pd.DataFrame()
#Now we take the weighted average
recommendation_df['weighted average recommendation score'] = tempTopUsersRating['sum_weightedRating']/tempTopUsersRating['sum_similarityIndex']
recommendation_df['movieId'] = tempTopUsersRating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,2.041667,1
2,1.968750,2
3,2.571429,3
4,1.875000,4
5,2.857143,5


In [18]:
recommendation_df = recommendation_df.sort_values(by='weighted average recommendation score', ascending=False)
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
2208,5.0,2208
5500,5.0,5500
81819,5.0,81819
2621,5.0,2621
6001,5.0,6001
182,5.0,182
6104,5.0,6104
6301,5.0,6301
506,5.0,506


In [ ]:
movie.loc[movie['movieId'].isin(recommendation_df.head(5)['movieId'].tolist())]